# Merged: FLAN-T5 Base Inspection + Training (Kaggle / T4, Internet On)

This notebook merges `flan-t5-base-inspection.ipynb` and `training.ipynb` into one runnable flow.
Core logic from both notebooks is preserved; only redundant/duplicate code paths were consolidated or renamed to avoid collisions.

In [ ]:
# Optional: install/upgrade dependencies (Kaggle usually already has these)
INSTALL_DEPS = False
if INSTALL_DEPS:
    %pip -q install --upgrade transformers datasets sentencepiece accelerate

In [ ]:
import re
import math
from collections import defaultdict, Counter

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import DataCollatorForSeq2Seq

In [ ]:
# Execution switches (keep heavy sections off by default for Kaggle run stability)
RUN_MODEL_DEMOS = True
RUN_MODEL_INSPECTION_PRINTS = True
RUN_FULL_FFN_TRAINABILITY_DEMO = True
RUN_TINY_DEMO_TRAINING = True
RUN_LORA_DUMMY_TRAINING = True

# Keep dataset print/debug cells optional
RUN_DATASET_INSPECTION = False

# PubMedQA sections (these are the heaviest)
RUN_PUBMEDQA_PIPELINE_SIMPLE = False
RUN_PUBMEDQA_TRAINING_WITH_ACCURACY = False

# 1. Model Loading

In [ ]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to(device)

print("Model loaded on:", device)

In [ ]:
if RUN_MODEL_DEMOS:
    model.eval()

    prompt = "What can you do"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128
        )

    print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

In [ ]:
if RUN_MODEL_DEMOS:
    sample_input = "Question: Is water wet?? Instruction: Answer in one sentence."
    inputs = tokenizer(sample_input, return_tensors="pt").to(device)

    outputs = model.generate(**inputs, max_new_tokens=64)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# 2. Model Inspection

In [ ]:
if RUN_MODEL_INSPECTION_PRINTS:
    print(model)

In [ ]:
if RUN_MODEL_INSPECTION_PRINTS:
    encoder_block_0 = model.encoder.block[0]
    print(encoder_block_0)

In [ ]:
if RUN_MODEL_INSPECTION_PRINTS:
    decoder_block_0 = model.decoder.block[0]
    print(decoder_block_0)

In [ ]:
if RUN_MODEL_INSPECTION_PRINTS:
    lm_head = model.lm_head
    print(lm_head)

In [ ]:
if RUN_MODEL_INSPECTION_PRINTS:
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params

    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters:    {frozen_params:,}")

# 3. Full Training (FFN-only trainability demo + tiny demo training)

In [ ]:
def freeze_all_params(model):
    for p in model.parameters():
        p.requires_grad = False


def enable_ffn_training(model):
    """
    Enable training only for FFN (DenseReluDense) layers
    in both encoder and decoder.
    """
    for name, module in model.named_modules():
        if module.__class__.__name__ == "T5DenseGatedActDense":
            for p in module.parameters():
                p.requires_grad = True


def print_trainable_params(model):
    for name, p in model.named_parameters():
        if p.requires_grad:
            print(name)


def count_parameters(model):
    total = 0
    trainable = 0
    for p in model.parameters():
        n = p.numel()
        total += n
        if p.requires_grad:
            trainable += n
    return total, trainable

In [ ]:
if RUN_FULL_FFN_TRAINABILITY_DEMO:
    param_stats = defaultdict(int)

    for name, param in model.named_parameters():
        param_stats[name.split('.')[0]] += param.numel()

    for k, v in param_stats.items():
        print(f"{k}: {v:,}")

In [ ]:
if RUN_FULL_FFN_TRAINABILITY_DEMO:
    for name, param in model.named_parameters():
        if "DenseReluDense" in name:
            print("FFN:", name, param.numel())
        elif "SelfAttention" in name or "EncDecAttention" in name:
            print("ATTN:", name, param.numel())

In [ ]:
if RUN_FULL_FFN_TRAINABILITY_DEMO:
    freeze_all_params(model)
    enable_ffn_training(model)
    print(sum(p.requires_grad for p in model.parameters()))

    print_trainable_params(model)

    total_params, trainable_params = count_parameters(model)
    percent = 100 * trainable_params / total_params

    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Trainable %: {percent:.2f}%")

In [ ]:
if RUN_FULL_FFN_TRAINABILITY_DEMO:
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params

    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters:    {frozen_params:,}")

In [ ]:
if RUN_FULL_FFN_TRAINABILITY_DEMO:
    sample_input = "Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences."
    inputs = tokenizer(sample_input, return_tensors="pt").to(device)

    outputs = model.generate(**inputs, max_new_tokens=64)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Demo training

In [ ]:
if RUN_TINY_DEMO_TRAINING:
    data = [
        {
            "instruction": "Question: Does aspirin reduce fever? Answer yes or no.",
            "output": "Yes, aspirin can reduce fever."
        },
        {
            "instruction": "Question: Is vitamin C a cure for cancer? Answer yes or no.",
            "output": "No, vitamin C is not a cure for cancer."
        }
    ]

In [ ]:
class SimpleT5Dataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        enc = self.tokenizer(
            item["instruction"],
            truncation=True,
            padding=False
        )

        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(
                item["output"],
                truncation=True,
                padding=False,
            )["input_ids"]

        labels = [
            l if l != self.tokenizer.pad_token_id else -100
            for l in labels
        ]

        return {
            "input_ids": torch.tensor(enc["input_ids"]),
            "attention_mask": torch.tensor(enc["attention_mask"]),
            "labels": torch.tensor(labels)
        }

In [ ]:
if RUN_TINY_DEMO_TRAINING:
    demo_dataset = SimpleT5Dataset(data, tokenizer)

    demo_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=-100
    )

    demo_train_loader = DataLoader(
        demo_dataset,
        batch_size=2,
        shuffle=True,
        collate_fn=demo_collator
    )

In [ ]:
if RUN_TINY_DEMO_TRAINING:
    model = model.float().to(device)

    from torch.optim import AdamW

    demo_optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-5
    )

In [ ]:
if RUN_TINY_DEMO_TRAINING:
    num_epochs = 3

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        for step, batch in enumerate(demo_train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}

            demo_optimizer.zero_grad()

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )

            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                max_norm=1.0
            )

            demo_optimizer.step()

            total_loss += loss.item()
            print(f"Step {step} | Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(demo_train_loader)
        print(f"Epoch {epoch + 1} average loss: {avg_loss:.4f}")

# 4. LoRA Training

## Apply LoRA to FFNs

In [ ]:
def apply_lora_to_ffn(model, r=8, alpha=1.0):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            continue  # Only wrap higher-level FFN
        if module.__class__.__name__ == "T5DenseGatedActDense":
            module.wi_0 = LoRALinear(module.wi_0, r=r, alpha=alpha)
            module.wi_1 = LoRALinear(module.wi_1, r=r, alpha=alpha)
            module.wo = LoRALinear(module.wo, r=r, alpha=alpha)


def merge_lora_to_linear(model):
    for name, module in model.named_modules():
        # Only target LoRALinear
        if isinstance(module, LoRALinear):
            # Create a new nn.Linear with the same shape
            new_linear = nn.Linear(
                module.base.in_features,
                module.base.out_features,
                bias=(module.base.bias is not None)
            ).to(module.base.weight.device)

            # Copy base weight + LoRA contribution
            new_linear.weight.data = module.base.weight.data + (module.B.weight @ module.A.weight) * module.scaling

            if module.base.bias is not None:
                new_linear.bias.data = module.base.bias.data

            # Replace LoRALinear in the parent module
            parent = module._modules
            for key, child in parent.items():
                if child is module:
                    parent[key] = new_linear
                    break


class LoRALinear(nn.Module):
    def __init__(self, base_linear, r=8, alpha=1.0):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False  # Freeze base

        in_dim = base_linear.in_features
        out_dim = base_linear.out_features

        self.A = nn.Linear(in_dim, r, bias=False)
        self.B = nn.Linear(r, out_dim, bias=False)
        self.scaling = alpha / r

        # Initialize LoRA
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scaling * self.B(self.A(x))

    # Expose weight/bias to satisfy HF generate()
    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias


# Temporarily compute effective weights for generation
def enable_lora_forward(model):
    for module in model.modules():
        if isinstance(module, LoRALinear):
            # Save original forward
            module._original_forward = module.forward
            # Override forward to include LoRA contribution
            module.forward = lambda x, m=module: m.base(x) + m.scaling * m.B(m.A(x))


# Restore original forward for training
def disable_lora_forward(model):
    for module in model.modules():
        if isinstance(module, LoRALinear) and hasattr(module, "_original_forward"):
            module.forward = module._original_forward
            del module._original_forward

## Dummy Training

In [ ]:
if RUN_LORA_DUMMY_TRAINING:
    rank = 16
    alpha = 32

    freeze_all_params(model)
    apply_lora_to_ffn(model, r=rank, alpha=alpha)

    for name, param in model.named_parameters():
        if "A.weight" in name or "B.weight" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

In [ ]:
if RUN_LORA_DUMMY_TRAINING:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable

    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters (LoRA): {trainable:,}")
    print(f"Frozen parameters: {frozen:,}")
    print(f"Trainable %: {100 * trainable / total:.4f}%")

In [ ]:
if RUN_LORA_DUMMY_TRAINING:
    class DummyDataset(Dataset):
        def __init__(self, tokenizer):
            self.data = [
                {"input": "Question: Is the sky blue? Instruction: Answer in one sentence.",
                 "output": "Yes, the sky is blue."},
                {"input": "Question: Do cats bark? Instruction: Answer in one sentence.",
                 "output": "No, cats do not bark."},
                {"input": "Question: Is water wet? Instruction: Answer in one sentence.",
                 "output": "Yes, water is wet."}
            ]
            self.tokenizer = tokenizer

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            example = self.data[idx]
            input_enc = self.tokenizer(example["input"], return_tensors="pt", truncation=True, padding=False)
            with self.tokenizer.as_target_tokenizer():
                labels = self.tokenizer(example["output"], return_tensors="pt", truncation=True, padding=False)["input_ids"]
            labels[labels == self.tokenizer.pad_token_id] = -100

            return {
                "input_ids": input_enc["input_ids"].squeeze(0),
                "attention_mask": input_enc["attention_mask"].squeeze(0),
                "labels": labels.squeeze(0)
            }

    dummy_train_dataset = DummyDataset(tokenizer)
    dummy_train_loader = DataLoader(dummy_train_dataset, batch_size=1, shuffle=True)

In [ ]:
if RUN_LORA_DUMMY_TRAINING:
    dummy_optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    model.to(device)
    model.train()

    num_epochs = 3

    for epoch in range(num_epochs):
        running_loss = 0.0
        for step, batch in enumerate(dummy_train_loader, 1):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            dummy_optimizer.zero_grad()
            loss.backward()
            dummy_optimizer.step()

            running_loss += loss.item()
            print(f"Step {step} | Loss: {loss.item():.4f}", end="\r")

        avg_loss = running_loss / len(dummy_train_loader)
        print(f"\nEpoch {epoch+1} average loss: {avg_loss:.4f}")

## Inference after training

In [ ]:
if RUN_LORA_DUMMY_TRAINING:
    merge_lora_to_linear(model)  # Always merge before inference

    model.to(device)
    model.eval()

    sample_input = "Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences."
    inputs = tokenizer(sample_input, return_tensors="pt").to(device)

    outputs = model.generate(**inputs, max_new_tokens=64)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
if RUN_LORA_DUMMY_TRAINING:
    sample_input = "Question: Is water wet?? Instruction: Answer in one sentence."
    inputs = tokenizer(sample_input, return_tensors="pt").to(device)

    outputs = model.generate(**inputs, max_new_tokens=64)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# Training using PubMedQA DS

In [ ]:
# Variant A (from inspection notebook)
def preprocess_pubmedqa_basic(example):
    context = " ".join(example["context"]["contexts"])

    input_text = (
        f"Question: {example['question']} "
        f"Context: {context} "
        f"Instruction: First answer yes, no, or maybe. "
        f"Then justify your answer briefly."
    )

    target_text = (
        f"Answer: {example['final_decision']}. "
        f"Explanation: {example['long_answer']}"
    )

    return {
        "input": input_text,
        "output": target_text,
    }



# Variant B (from training notebook; keeps short_answer for evaluation)
def preprocess_pubmedqa_with_short_answer(example):
    context = " ".join(example["context"]["contexts"])

    input_text = (
        f"Question: {example['question']} "
        f"Context: {context} "
        f"Instruction: Answer yes, no, or maybe. Then justify your answer."
    )

    target_text = (
        f"Answer: {example['final_decision']}. "
        f"Explanation: {example['long_answer']}"
    )
    short_answer = example["final_decision"]

    return {
        "input": input_text,
        "output": target_text,
        "short_answer": short_answer,
    }



max_input_length = 512
max_output_length = 128
max_target_length = max_output_length  # compatibility alias


def tokenize_for_t5(example):
    # Encode inputs
    input_enc = tokenizer(
        example["input"],
        truncation=True,
        padding="max_length",
        max_length=max_input_length,
    )

    # Encode targets
    target_enc = tokenizer(
        example["output"],
        truncation=True,
        padding="max_length",
        max_length=max_output_length,
    )

    labels = target_enc["input_ids"]
    labels = [l if l != tokenizer.pad_token_id else -100 for l in labels]

    return {
        "input_ids": input_enc["input_ids"],
        "attention_mask": input_enc["attention_mask"],
        "labels": labels,
    }



def collate_fn(batch):
    input_ids = torch.tensor([item["input_ids"] for item in batch], dtype=torch.long)
    attention_mask = torch.tensor([item["attention_mask"] for item in batch], dtype=torch.long)
    labels = torch.tensor([item["labels"] for item in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

## PubMedQA pipeline (simple)

In [ ]:
if RUN_PUBMEDQA_PIPELINE_SIMPLE:
    dataset = load_dataset("pubmed_qa", "pqa_labeled")
    if RUN_DATASET_INSPECTION:
        print(dataset)
        print(dataset.column_names)
        print(dataset["train"].features)
        print(dataset["train"][0])
        print(dataset["train"][0]["final_decision"])
    print(set(dataset["train"]["final_decision"]))
    print(Counter(dataset["train"]["final_decision"]))

    processed_ds = dataset.map(
        preprocess_pubmedqa_basic,
        remove_columns=dataset["train"].column_names,
    )
    if RUN_DATASET_INSPECTION:
        print(processed_ds["train"][0])

    tokenized_dataset = processed_ds.map(
        tokenize_for_t5,
        remove_columns=processed_ds["train"].column_names,
    )

    simple_batch_size = 2  # small for testing, increase if GPU allows
    simple_train_loader = DataLoader(
        tokenized_dataset["train"],
        batch_size=simple_batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    if RUN_DATASET_INSPECTION:
        print(next(iter(simple_train_loader)))

    rank = 128
    alpha = 256

    freeze_all_params(model)
    apply_lora_to_ffn(model, r=rank, alpha=alpha)

    for name, param in model.named_parameters():
        if "A.weight" in name or "B.weight" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable

    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters (LoRA): {trainable:,}")
    print(f"Frozen parameters: {frozen:,}")
    print(f"Trainable %: {100 * trainable / total:.4f}%")

    model.to(device)
    simple_optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3,
    )

    num_epochs = 3
    model.train()

    for epoch in range(num_epochs):
        running_loss = 0.0
        loop = tqdm(enumerate(simple_train_loader, 1), total=len(simple_train_loader), desc=f"Epoch {epoch+1}")

        for step, batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            simple_optimizer.zero_grad()
            loss.backward()
            simple_optimizer.step()

            running_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_loss = running_loss / len(simple_train_loader)
        print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

    merge_lora_to_linear(model)
    model.eval()

    sample_input = "Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences."
    inputs = tokenizer(sample_input, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=64)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## PubMedQA training (with short-answer accuracy eval)

In [ ]:
if RUN_PUBMEDQA_TRAINING_WITH_ACCURACY:
    dataset = load_dataset("pubmed_qa", "pqa_labeled")
    if RUN_DATASET_INSPECTION:
        print(dataset)
        print(dataset["train"].features)
        print(dataset["train"][0])

    dataset = dataset.map(
        preprocess_pubmedqa_with_short_answer,
        remove_columns=dataset["train"].column_names,
    )
    if RUN_DATASET_INSPECTION:
        print(dataset["train"][0])

    split_datasets = dataset["train"].train_test_split(test_size=0.1, seed=42)

    train_data = split_datasets["train"]
    val_data = split_datasets["test"]

    if RUN_DATASET_INSPECTION:
        print(train_data)
        print(val_data)

    train_data = train_data.remove_columns("short_answer")
    val_short_answers = [ex["short_answer"] for ex in val_data]
    val_data = val_data.remove_columns("short_answer")

    if RUN_DATASET_INSPECTION:
        print(len(val_short_answers))

    train_dataset = train_data.map(tokenize_for_t5, remove_columns=train_data.column_names)
    val_dataset = val_data.map(tokenize_for_t5, remove_columns=val_data.column_names)

    batch_size = 8
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )
    if RUN_DATASET_INSPECTION:
        print(next(iter(train_loader)))

    rank = 128
    alpha = 256

    freeze_all_params(model)
    apply_lora_to_ffn(model, r=rank, alpha=alpha)

    for name, param in model.named_parameters():
        if "A.weight" in name or "B.weight" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable

    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters (LoRA): {trainable:,}")
    print(f"Frozen parameters: {frozen:,}")
    print(f"Trainable %: {100 * trainable / total:.4f}%")

    model.to(device)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3,
    )

    num_epochs = 3
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        loop = tqdm(enumerate(train_loader, 1), total=len(train_loader), desc=f"Epoch {epoch+1}")

        for step, batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

        # Evaluation: compute short-answer accuracy
        model.eval()
        enable_lora_forward(model)  # temporarily enable LoRA for generation

        correct = 0
        total = 0
        global_idx = 0  # to index val_short_answers

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Evaluating"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)

                outputs = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=128,
                    do_sample=False
                )

                gen_texts = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

                for text in gen_texts:
                    match = re.search(r"Answer:\s*(yes|no|maybe)", text, re.IGNORECASE)
                    pred = match.group(1).lower() if match else ""

                    true_label = val_short_answers[global_idx].lower()
                    if pred == true_label:
                        correct += 1
                    total += 1
                    global_idx += 1

        accuracy = correct / total if total > 0 else 0.0
        print(f"Epoch {epoch+1} short-answer accuracy: {accuracy:.4f}")

        disable_lora_forward(model)  # restore training forward

    # Generate a few samples on val set
    enable_lora_forward(model)
    model.eval()

    generated_texts = []
    raw_val_texts = [ex["input"] for ex in val_data]
    raw_val_labels = [ex["output"] for ex in val_data]

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Generating on val set"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=128,
                do_sample=False
            )

            batch_texts = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
            generated_texts.extend(batch_texts)

    for i in range(5):  # first 5 examples
        print(f"Example {i+1}:")
        print(f"Question + Context + Instruction:\n{raw_val_texts[i]}\n")
        print(f"True Short Answer: {raw_val_labels[i]}")
        print(f"Generated Answer:\n{generated_texts[i]}\n")
        print("-" * 80)

    disable_lora_forward(model)